In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time
import pandas as pd
import traceback

In [25]:

INPUT_XLSX = r"D:\BaiDoAnChuyenNganh3\Automated-Resume-Ranking-System-main\csvfiles\crawlcv\resumeworded_links.xlsx"
OUTPUT_XLSX = r"D:\BaiDoAnChuyenNganh3\Automated-Resume-Ranking-System-main\csvfiles\crawlcv\resumeworded_final.xlsx"

WAIT_LONG = 5
WAIT_SHORT = 1.0

In [26]:
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
# options.add_argument("--headless=new")

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, WAIT_LONG)

In [27]:
def safe_scroll_into_view(el):
    try:
        driver.execute_script("arguments[0].scrollIntoView({block: 'center', inline: 'center'});", el)
        time.sleep(0.4)
    except Exception:
        pass

def safe_js_click(el):
    """Click bằng JS để tránh trường hợp bị overlay"""
    driver.execute_script("arguments[0].click();", el)

In [28]:
df_links = pd.read_excel(INPUT_XLSX)

In [29]:
results = []

In [30]:
for idx, row in df_links.iterrows():
    src_link = str(row.get("Links", "")).strip()
    category = str(row.get("Category", "")).strip()

    if not src_link:
        continue

    print(f"\n=== ({idx+1}/{len(df_links)}) Processing link: {src_link} | Category: {category}")

    try:
        driver.get(src_link)
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        time.sleep(2)
        full_scroll()
    except Exception as e:
        print(" -> Lỗi khi mở link:", e)
        continue

    try:
        top_bar_elems = driver.find_elements(By.CSS_SELECTOR, "div.template-image-container-top-bar")
    except Exception as e:
        print(" -> Không tìm thấy top-bar nào:", e)
        top_bar_elems = []

    print(f" -> Tìm thấy {len(top_bar_elems)} top-bar(s).")

    for tb_index, tb in enumerate(top_bar_elems, start=1):
        try:
            print(f"   - Xử lý top-bar #{tb_index}/{len(top_bar_elems)} ...")
            safe_scroll_into_view(tb)
            time.sleep(WAIT_SHORT)

            try:
                pdf_tab = tb.find_element(By.CSS_SELECTOR, "div.tab-elemnt-template-container.pdf-tab")
            except Exception:
                pdf_tab = None

            try:
                screenshot_tab = tb.find_element(By.CSS_SELECTOR, "div.tab-elemnt-template-container.screenshot-tab")
            except Exception:
                screenshot_tab = None

            if pdf_tab:
                safe_scroll_into_view(pdf_tab)
                safe_js_click(pdf_tab)
                print("     -> Đã click vào pdf-tab (Text Version), đợi load iframe...")
                time.sleep(WAIT_LONG)

                iframe_texts = []
                iframes = driver.find_elements(By.TAG_NAME, "iframe")
                print(f"     -> Tìm thấy {len(iframes)} iframe(s).")

                for iframe in iframes:
                    try:
                        driver.switch_to.frame(iframe)
                        p_elems = driver.find_elements(By.TAG_NAME, "p")
                        if p_elems:
                            texts = [p.text.strip() for p in p_elems if p.text.strip()]
                            if texts:
                                iframe_texts.extend(texts)
                        driver.switch_to.default_content()
                    except Exception:
                        driver.switch_to.default_content()
                        continue

                if not iframe_texts:
                    try:
                        content_blocks = driver.find_elements(
                            By.CSS_SELECTOR,
                            "div.ndfHFb-c4YZDc-cYSp0e-DARUcf-Df1ZY-bN97Pc-haAclf"
                        )
                        for cb in content_blocks:
                            p_elems = cb.find_elements(By.TAG_NAME, "p")
                            for p in p_elems:
                                text = p.text.strip()
                                if text:
                                    iframe_texts.append(text)
                    except Exception:
                        pass

                paragraphs_combined = " | ".join(iframe_texts).strip()
                print(f"     -> Done top-bar #{tb_index}: lấy được {len(iframe_texts)} đoạn.")

                results.append({
                    "Source_Link": src_link,
                    "Category": category,
                    "TopBar_Index": tb_index,
                    "Paragraphs": paragraphs_combined
                })

                if screenshot_tab:
                    try:
                        safe_scroll_into_view(screenshot_tab)
                        safe_js_click(screenshot_tab)
                        print("     -> Đã chuyển lại về screenshot-tab.")
                        time.sleep(1.5)
                    except Exception as e_back:
                        print("     -> Lỗi khi chuyển lại screenshot-tab:", e_back)
                else:
                    print("     -> Không tìm thấy screenshot-tab để chuyển lại.")

            else:
                print("     -> Không tìm thấy pdf-tab trong top-bar này. Bỏ qua.")

            time.sleep(1.2)

        except Exception as e_tb:
            print("   -> Lỗi khi xử lý top-bar:", e_tb)
            traceback.print_exc()
            results.append({
                "Source_Link": src_link,
                "Category": category,
                "TopBar_Index": tb_index,
                "Paragraphs": "",
                "Num_Paragraphs": 0,
                "Error": str(e_tb)
            })

    print(f" -> ✅ Hoàn tất link: {src_link} (đã xử lý {len(top_bar_elems)} top-bar).")
    time.sleep(2)


=== (1/74) Processing link: https://resumeworded.com/data-analyst-resume-examples | Category: Data Analyst Resume Guide & Examples for 2025
 -> Tìm thấy 19 top-bar(s).
   - Xử lý top-bar #1/19 ...
     -> Đã click vào pdf-tab (Text Version), đợi load iframe...
     -> Tìm thấy 14 iframe(s).
     -> Done top-bar #1: lấy được 83 đoạn.
     -> Đã chuyển lại về screenshot-tab.
   - Xử lý top-bar #2/19 ...
     -> Đã click vào pdf-tab (Text Version), đợi load iframe...
     -> Tìm thấy 15 iframe(s).
     -> Done top-bar #2: lấy được 45 đoạn.
     -> Đã chuyển lại về screenshot-tab.
   - Xử lý top-bar #3/19 ...
     -> Đã click vào pdf-tab (Text Version), đợi load iframe...
     -> Tìm thấy 16 iframe(s).
     -> Done top-bar #3: lấy được 87 đoạn.
     -> Đã chuyển lại về screenshot-tab.
   - Xử lý top-bar #4/19 ...
     -> Đã click vào pdf-tab (Text Version), đợi load iframe...
     -> Tìm thấy 17 iframe(s).
     -> Done top-bar #4: lấy được 44 đoạn.
     -> Đã chuyển lại về screenshot-tab.

In [31]:
print(results)

[{'Source_Link': 'https://resumeworded.com/data-analyst-resume-examples', 'Category': 'Data Analyst Resume Guide & Examples for 2025', 'TopBar_Index': 1, 'Paragraphs': "First Last | Data Analyst | WORK EXPERIENCE | ______________________________________________________________________ | Resume Worded, London, United Kingdom | Education technology startup with 50+ employees and $100m+ annual revenue | Data Analyst 08/2021 – Present | ● Managed large datasets with 20K observations using regular expressions | and selecting key variables to build models for statistical logic. | ● Created explanatory models of use cases for presentation to 100+ project | stakeholders and 40+ non-technical audiences. | ● Performed data cleaning necessary for predictive models by assessing | information from 1200+ customers in the construction and financial sectors. | ● Collaborated with 30+ data analysts, data scientists, and project managers | on several concurrent projects in Q1 2021. | Polyhire, London, U

In [32]:
driver.quit()

In [33]:
if results:
    df = pd.DataFrame(results)
    df.to_excel(OUTPUT_XLSX, index=False)
    print(f"\n✅ Crawl hoàn tất! Đã lưu {len(results)} mục vào:\n{OUTPUT_XLSX}")
else:
    print("⚠️ Không có dữ liệu nào được crawl!")


✅ Crawl hoàn tất! Đã lưu 531 mục vào:
D:\BaiDoAnChuyenNganh3\Automated-Resume-Ranking-System-main\csvfiles\crawlcv\resumeworded_final.xlsx


In [6]:
import pandas as pd
import re

# ====== Đường dẫn ======
INPUT_FILE = r"D:\BaiDoAnChuyenNganh3\Automated-Resume-Ranking-System-main\csvfiles\crawlcv\resumeworded_final.xlsx"
OUTPUT_FILE = r"D:\BaiDoAnChuyenNganh3\Automated-Resume-Ranking-System-main\csvfiles\crawlcv\resumeworded_cv.xlsx"

# ====== Đọc file Excel ======
df = pd.read_excel(INPUT_FILE)

# ====== Xử lý cột Category ======
def clean_category(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.replace(" Resume Guide & Examples for 2025", "").strip()
    return text

# ====== Xử lý cột Resume ======
def clean_resume(text):
    if pd.isna(text):
        return ""
    text = str(text)

    # 1. Xóa "FIRST LAST" ở đầu văn bản (không phân biệt hoa thường)
    text = re.sub(r'^\s*first\s+last\s*', '', text, flags=re.IGNORECASE)

    # 2. Xóa các ký tự / chuỗi không cần thiết
    text = text.replace("|", " ")
    text = text.replace("•", " ")
    text = text.replace("●", " ")
    text = re.sub(r'_+', ' ', text)  # xóa chuỗi nhiều dấu gạch dưới

    # 3. Chuẩn hóa khoảng trắng
    text = re.sub(r'\s+', ' ', text).strip()

    # 4. Bỏ xuống dòng (đưa về 1 dòng)
    text = text.replace("\n", " ").replace("\r", " ")

    return text

# ====== Áp dụng làm sạch ======
if "Category" in df.columns:
    df["Category"] = df["Category"].apply(clean_category)
if "Resume" in df.columns:
    df["Resume"] = df["Resume"].apply(clean_resume)




In [7]:
print(df.head())

       Category                                             Resume
0  Data Analyst  Data Analyst WORK EXPERIENCE Resume Worded, Lo...
1  Data Analyst  Bay Area, California +1-234-456-789 profession...
2  Data Analyst  Business Data Analyst Billings, Montana +1-234...
3  Data Analyst  Power BI Data Analyst Omaha, Nebraska +1-234-4...
4  Data Analyst  Data Analyst Intern WORK EXPERIENCE Resume Wor...


In [8]:
# ====== Lưu lại file Excel mới ======
df.to_excel(OUTPUT_FILE, index=False, engine='openpyxl')
print(f"✅ Đã làm sạch dữ liệu và lưu vào: {OUTPUT_FILE}")

✅ Đã làm sạch dữ liệu và lưu vào: D:\BaiDoAnChuyenNganh3\Automated-Resume-Ranking-System-main\csvfiles\crawlcv\resumeworded_cv.xlsx
